# Chequeos de cuadratura y privacidad

Verifica el total publicado de Cochilco, los conteos del corte SEA, la ausencia de columnas personales y la completitud de fechas necesarias para el análisis.

In [ ]:
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

from cmip.extract.sea import PRIVATE_COLUMNS

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
cochilco = pd.read_parquet(ROOT / 'data/processed/cochilco_projects.parquet')
sea_projects = pd.read_parquet(ROOT / 'data/interim/sea_projects.parquet')
sea_mining = pd.read_parquet(ROOT / 'data/interim/sea_mining.parquet')

In [ ]:
expected_events = {
    'aprobado': 1057,
    'desistido_o_abandonado': 372,
    'no_admitido': 316,
    'termino_anticipado': 122,
    'en_tramite': 58,
    'rechazado': 36,
}
duracion_proyectos = sea_projects.loc[
    ~sea_projects.fecha_inconsistente, 'duracion_dias'
]
duracion_mineria = sea_mining.loc[
    ~sea_mining.fecha_inconsistente, 'duracion_dias'
]
checks = {
    'Cochilco: 59 filas': len(cochilco) == 59,
    'Cochilco: 104.549,2 MMUS$': abs(cochilco.inversion_musd.sum() - 104_549.2) < 1e-9,
    'SEA proyectos: 13.735': len(sea_projects) == 13_735,
    'SEA minería: 1.961': len(sea_mining) == 1_961,
    'SEA minería: conteos de evento': sea_mining.evento.value_counts().to_dict() == expected_events,
    'Sin columnas personales en SEA proyectos': PRIVATE_COLUMNS.isdisjoint(sea_projects.columns),
    'Sin columnas personales en SEA minería': PRIVATE_COLUMNS.isdisjoint(sea_mining.columns),
    'Fecha de ingreso completa en SEA proyectos': sea_projects.fecha_ingreso.notna().all(),
    'Fecha de ingreso completa en SEA minería': sea_mining.fecha_ingreso.notna().all(),
    'Duración completa si la fecha es consistente (proyectos)': duracion_proyectos.notna().all(),
    'Duración completa si la fecha es consistente (minería)': duracion_mineria.notna().all(),
}
resultado = pd.Series(checks, name='cumple').rename_axis('chequeo').to_frame()
display(resultado)
assert resultado.cumple.all(), resultado.loc[~resultado.cumple]

In [ ]:
display(
    sea_mining.evento.value_counts()
    .rename_axis('evento')
    .rename('n')
    .to_frame()
)
print(f'Inversión Cochilco: {cochilco.inversion_musd.sum():,.1f} MMUS$')